# S6E10 | Stacking 201 public OOF members, then blending with the best public stacks

Many people have shared out-of-fold (OOF) predictions for this episode. [Kratosyan's notebook](https://www.kaggle.com/code/kratosyan/s6e10-stacking-every-public-oof-library)
showed that one logistic-regression stack over every public library beats each individual stack (nested CV 0.96205, LB 0.96167).

v5 adds a final blend with the best public stack outputs. v9 simplifies that blend to 0.95 × the samanyu1808 Megastack output + 0.05 × my [Trust the CV](https://www.kaggle.com/code/yeashusemwal/trust-the-cv) stack (section 6).

This notebook extends that idea (**v2** added Chris Deotte's tabular-foundation-model members and a stronger L2 penalty; **v3** adds three new members of dariushafshar's library and submits the plain LR stack):

| step | members | nested CV |
|---|---:|---:|
| kratosyan's 124 members, LR(C=1) (reproduced) | 124 | 0.962048 |
| + 10 newer members (15-fold TabPFN-3.5 ×3, 15-fold CatBoost/XGB, 10-fold route-teacher XGB/RealMLP, CatBoost numeric-CTR, …) | 134 | 0.962101 |
| + every other public OOF library and stack-level OOFs, C=0.01 | 187 | 0.962120 |
| stronger L2 penalty, C=0.003 | 187 | 0.962134 |
| **+ 9 tabular foundation models + tuned XGBoost/RealMLP ([Chris Deotte](https://www.kaggle.com/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions)) + 1 new sachith7 member** | 199 | **0.962170** |
| 0.8 × LR stack + 0.2 × LightGBM stacker | 199 | 0.962180 |
| **v3: + 3 new golem members (full-context TabPFN-3.5, LightGBM + aux-task features, its 10-fold version)** | 202 | **0.962175** |
| LR + LightGBM blends (comparison only, not submitted) | 202 | see section 5 |

**v3 and v4 submit the plain LR stack (C = 0.003; v4 has 201 members, public LB 0.96181).** Since v4 every library is optional: Den Pugovkin's CatBoost numeric-CTR member is no longer in his notebook's latest output, so this version stacks 201 members. The exact numbers are printed below. The LightGBM blend is still computed below as a comparison: its nested
CV was about 0.00001 higher in our local runs (well inside the noise), but its weight is picked on the same OOF, while the plain LR has no tuned blend weight. On the public
LB the plain LR has also come out ahead in 4 of 4 paired submissions (by ≤ 0.00003, i.e. noise either way).

What moved the score: the **tabular foundation models** (TabPFN-3.5, Kumo, TabICL-2, Mitra, LimiX, …: +0.00003 even on top of 187 members),
the **15-fold × 3-seed TabPFN-3.5 members** by AbdullahSafwan333 (+0.00004), a strong L2 penalty (C=0.003, +0.00003 over C=1), and
blending a LightGBM stacker into the LR stack (+0.00001). Members from yet more GBDT/NN libraries on the same features add ≈0.

**Honest-validation notes.** Every member's OOF was produced by its author on their own folds (mostly `StratifiedKFold(5, shuffle=True, random_state=42)`;
some use 10/15 folds or seed 33; every row is still out of fold for that member). The stackers themselves are scored with a second, nested CV
(meta folds `StratifiedKFold(5, shuffle=True, random_state=7)`). The 0.8/0.2 weight was chosen on the nested OOF (one parameter, so slightly optimistic).
Public LB sits about 0.0004 below nested CV, and on public-LB-sized subsets the SD between two strong stacks is about 1e-4, so choose by CV, not LB.
(v1, 187 members: nested CV 0.96215, public LB 0.96171. v2, 199 members, LR+LightGBM blend: nested CV 0.96218, public LB 0.96176.)

**Caveat on the v3 members** (disclosed by their author): `p_lgbm_auxev` and `q_lgbm_auxev_10fold` early-stop on the held-out fold, so their OOF is mildly optimistic, and their count/route-profile features are transductive (label-free). Their small nested-CV gain may be slightly inflated; the public LB moved the same way (0.96179 → 0.96180).

Nothing is trained on raw features from scratch here: all credit for the members goes to their authors (section 1). Please upvote them.

## 1. Sources (please upvote them)

| source | members |
|---|---|
| [kratosyan · Stacking every public OOF library](https://www.kaggle.com/code/kratosyan/s6e10-stacking-every-public-oof-library) | loader code and the 124-member set below |
| [goodpjw2008 · Part 8](https://www.kaggle.com/code/goodpjw2008/s6e10-tabpfn-route-categories-lb-0-96160) | 48 (GBDT/RealMLP/DeepFM on route-ID features + TabPFN-3.5/TabICL) |
| [sachith7 · stack OOF predictions](https://www.kaggle.com/datasets/sachith7/s6e10-stack-oof-predictions) | 22 (10 of them 10-fold) |
| [megayak · OOF library](https://www.kaggle.com/datasets/megayak/s6e10-oof-library), [honest OOF stack](https://www.kaggle.com/code/megayak/s6e10-honest-oof-stack) | 11 + 4 |
| [dariushafshar · golem OOF library](https://www.kaggle.com/datasets/dariushafshar/s6e10-golem-oof-library) | 14 |
| [arhancanli12 · OOF + test predictions](https://www.kaggle.com/datasets/arhancanli12/s6e10-oof-test-predictions) | 4 |
| [najiama · OOF](https://www.kaggle.com/datasets/najiama/s6e10-oof) | blends 01/02/03 |
| [samanyu1808 · TabPFN route+og](https://www.kaggle.com/code/samanyu1808/s6e10-tabpfn-route-og), [seeds](https://www.kaggle.com/code/samanyu1808/s6e10-tabpfn-seeds); [hemingweb · TabPFN-3.5 full context](https://www.kaggle.com/code/hemingweb/s6e10-exp01-tabpfn35-full-context) | 3 |
| [mitudru · equality blocks](https://www.kaggle.com/code/mitudru/s6e10-analysis-first-equality-blocks-stack) | 7 |
| [wangxintong111 · CatBoost rating crosses](https://www.kaggle.com/code/wangxintong111/s6e10-catboost-rating-crosses-cv-0-96080), [LightGBM context stats](https://www.kaggle.com/code/wangxintong111/s6e10-lightgbm-context-stats-cv-0-96065) | 2 |
| [thisray · XGB/RealMLP/CatBoost](https://www.kaggle.com/code/thisray/s6e10-xgb-realmlp-catboost-lb-0-96123) | 3 |
| [busyaprime · Route or distance? Both](https://www.kaggle.com/code/busyaprime/s6e10-route-or-distance-both-lb-0-96169) (members via [kratosyan's dataset](https://www.kaggle.com/datasets/kratosyan/s6e10-busyaprime-five-members-oof)) | 5 + stack OOF |
| **new:** [AbdullahSafwan333 · OOF dataset](https://www.kaggle.com/datasets/abdullahsafwan333/oof-dataset) ([TabPFN notebook](https://www.kaggle.com/code/abdullahsafwan333/aviator-tabpfn)) | 15-fold × 3-seed TabPFN-3.5 ×3, CatBoost, XGBoost |
| **new:** [kratosyan · XGBoost + RealMLP route ID + teacher](https://www.kaggle.com/code/kratosyan/s6e10-xgboost-realmlp-route-id-teacher) | 2 (10-fold) |
| **new:** [hermengardo · Single CatBoost](https://www.kaggle.com/code/hermengardo/single-catboost-eda-ps6e10) | 1 |
| **new:** [Den Pugovkin · Numeric Keys CatBoost CTR](https://www.kaggle.com/code/denpugovkin/s6e10-numeric-keys-catboost-ctr) | 1 |
| **new:** [Vladimir Demidov · RealMLP PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit) | 1 |
| **new:** [Sadam Torres · artifacts](https://www.kaggle.com/datasets/sadamtorres/ps-s6e10-airline-satisfaction-artifacts) | 38 + 2 original-data teachers |
| **new:** [Ahmet Kağan Koral · Encodings + CatBoost + RealMLP](https://www.kaggle.com/code/kagankoral/s6e10-encodings-catboost-realmlp-lb-0-961) | 5 |
| **new:** [Aman Atar · Original-Prior Stack v5](https://www.kaggle.com/code/amanatar/s6e10-the-original-prior-stack-v5) | 5 |
| **v2:** [Chris Deotte · TFM OOF and test predictions](https://www.kaggle.com/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions) ([notebook](https://www.kaggle.com/code/cdeotte/stacking-tfms-with-public-oof)) | 11: TabPFN-3.5, TabICL-2, Kumo large/small, Mitra-2, LimiX-2, TabFM, EXAONE, Causilo, tuned XGBoost, tuned RealMLP |
| **v3:** [dariushafshar · golem OOF library](https://www.kaggle.com/datasets/dariushafshar/s6e10-golem-oof-library) (now 17 models) | 3 new: `o_tabpfn35` (full-context TabPFN-3.5), `p_lgbm_auxev`, `q_lgbm_auxev_10fold` |
| **v2:** [sachith7 · stack OOF predictions](https://www.kaggle.com/datasets/sachith7/s6e10-stack-oof-predictions) | 1 new member (`xgb_A_auxall`), picked up automatically |

TabPFN-3.5 members come with Prior Labs' non-commercial licence; their authors note that it allows use in Kaggle competitions.
Kernel outputs are pinned to the versions attached here; if an author re-runs theirs, numbers can move in the 5th decimal.

In [1]:
import glob, os, time, json
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

T0 = time.time()
def find_dir(slug):
    hits = [p for p in glob.glob(f'/kaggle/input/**/{slug}', recursive=True) if os.path.isdir(p)]
    assert hits, f'input not attached: {slug}'
    return sorted(hits, key=len)[0]

COMP = find_dir('playground-series-s6e10')
tr, te = pd.read_csv(f'{COMP}/train.csv'), pd.read_csv(f'{COMP}/test.csv')
y = tr['satisfaction'].astype(int).values
NTR = len(y)
OOF, TEST, SRC = {}, {}, {}

def add(name, o, t, src):
    o, t = np.asarray(o, float), np.asarray(t, float)
    assert o.shape == (NTR,) and t.shape == (len(te),), (name, o.shape, t.shape)
    assert np.isfinite(o).all() and np.isfinite(t).all(), name
    OOF[name], TEST[name], SRC[name] = o, t, src

def by_id(df, ids, col):
    return df.set_index('id')[col].loc[ids].values

## 2. Load every library
Every array is aligned to `train.csv` / `test.csv` order by `id` (or asserted to be in that order).

In [2]:
# ---------------- kratosyan's 124-member set ----------------
def l_gp():
    d = find_dir('s6e10-tabpfn-route-categories-lb-0-96160')
    go, gt = pd.read_csv(f'{d}/oof_members.csv'), pd.read_csv(f'{d}/test_members.csv')
    assert (go['id'].values == tr['id'].values).all() and (gt['id'].values == te['id'].values).all()
    for c in go.columns[1:]:
        add('gp_' + c, go[c].values, gt[c].values, 'goodpjw P8')
def l_mg():
    d = find_dir('s6e10-oof-library')
    mo, mt = pd.read_parquet(f'{d}/oof_predictions.parquet'), pd.read_parquet(f'{d}/test_predictions.parquet')
    assert (mo['id'].values == tr['id'].values).all() and (mt['id'].values == te['id'].values).all()
    groups = {}
    for c in mo.columns:
        if c in ('id', 'fold', 'satisfaction'):
            continue
        groups.setdefault(c.rsplit('_s', 1)[0] if c.startswith('realmlp') else c, []).append(c)
    for g, cols in groups.items():
        add('mg_' + g, mo[cols].mean(1).values, mt[cols].mean(1).values, 'megayak')
def l_gl():
    d = find_dir('s6e10-golem-oof-library')
    for k in 'abcdefghijklmn':
        add('gl_' + k, np.load(f'{d}/oof_{k}.npy'), np.load(f'{d}/test_{k}.npy'), 'golem')
    for k in ['o_tabpfn35', 'p_lgbm_auxev', 'q_lgbm_auxev_10fold']:   # v3: added to the library on 2026-10-08
        add('gl_' + k, np.load(f'{d}/oof_{k}.npy'), np.load(f'{d}/test_{k}.npy'), 'golem')
def l_ar():
    d = find_dir('s6e10-oof-test-predictions')
    ao, at = pd.read_parquet(f'{d}/oof_predictions.parquet'), pd.read_parquet(f'{d}/test_predictions.parquet')
    assert (ao['id'].values == tr['id'].values).all() and (at['id'].values == te['id'].values).all()
    for c in ['lgb_5f', 'xgb_5f', 'cat_5f', 'realmlp_5f']:
        add('ar_' + c, ao[c].values, at[c].values, 'arhancanli12')
def l_sa():
    d = find_dir('s6e10-stack-oof-predictions')
    for f in sorted(glob.glob(f'{d}/*.npz')):
        z = np.load(f)
        add('sa_' + os.path.basename(f)[:-4], z['oof'], z['test'], 'sachith7')
def l_nj():
    d = find_dir('s6e10-oof')
    add('nj_01', by_id(pd.read_csv(f'{d}/01_blend_oof.csv'), tr['id'], 'satisfaction'),
        by_id(pd.read_csv(f'{d}/01_submission.csv'), te['id'], 'satisfaction'), 'najiama')
def l_pfn():
    for nm, slug in [('pfn_samanyu_route_og', 's6e10-tabpfn-route-og'), ('pfn_samanyu_seeds', 's6e10-tabpfn-seeds'),
                     ('pfn_hemingweb_raw', 's6e10-exp01-tabpfn35-full-context')]:
        d = find_dir(slug)
        add(nm, by_id(pd.read_csv(f'{d}/oof_exp01_tabpfn35.csv'), tr['id'], 'pred'),
            by_id(pd.read_csv(f'{d}/test_exp01_tabpfn35.csv'), te['id'], 'satisfaction'), 'TabPFN outputs')
def l_mi():
    d = find_dir('s6e10-analysis-first-equality-blocks-stack')
    mo2 = pd.read_parquet(f'{d}/oof_predictions.parquet').set_index('id').loc[tr['id']]
    mt2 = pd.read_parquet(f'{d}/test_predictions.parquet').set_index('id').loc[te['id']]
    for c in mo2.columns:
        add('mi_' + c, mo2[c].values, mt2[c].values, 'mitudru')
def l_wx():
    for slug, c in [('s6e10-catboost-rating-crosses-cv-0-96080', 'rating_segments'),
                    ('s6e10-lightgbm-context-stats-cv-0-96065', 'context_numeric')]:
        d = find_dir(slug)
        add('wx_' + c, by_id(pd.read_parquet(f'{d}/oof_predictions.parquet'), tr['id'], c),
            by_id(pd.read_parquet(f'{d}/test_predictions.parquet'), te['id'], c), 'wangxintong111')
def l_th():
    for slug, tag in [('s6e10-xgb-realmlp-catboost-lb-0-96123', 'thisray'), ('s6e10-honest-oof-stack', 'megayak honest')]:
        d = find_dir(slug)
        o = pd.read_csv(f'{d}/oof_members.csv').set_index('id').loc[tr['id']]
        t = pd.read_csv(f'{d}/test_members.csv').set_index('id').loc[te['id']]
        for c in t.columns:
            add(f'{tag.split()[0][:2]}{"h" if "honest" in tag else ""}_{c}', o[c].values, t[c].values, tag)
def l_bp():
    d = find_dir('s6e10-busyaprime-five-members-oof')
    for f in sorted(glob.glob(f'{d}/bp_*.npz')):
        z = np.load(f)
        add(os.path.basename(f)[:-4], z['oof'], z['test'], 'busyaprime')

# ---------------- new members (not in kratosyan v3) ----------------
def l_ab():  # abdullahsafwan333: 15-fold x 3-seed TabPFN-3.5 variants + CatBoost/XGB (15 folds)
    d = find_dir('oof-dataset')
    for m in ['tabpfn_catfd10', 'tabpfn_raw', 'tabpfn_te_cnt', 'cat_v3', 'xgb_v3']:
        add('ab_' + m, by_id(pd.read_csv(f'{d}/{m}_oof.csv'), tr['id'], 'pred'),
            by_id(pd.read_csv(f'{d}/{m}_test.csv'), te['id'], 'pred'), 'abdullah ' + ('pfn' if 'tabpfn' in m else 'gbdt'))
def l_kx():  # kratosyan 10-fold route + original-teacher + aux members
    d = find_dir('s6e10-xgboost-realmlp-route-id-teacher')
    for f in sorted(glob.glob(f'{d}/**/*_route_ogaux.npz', recursive=True)):
        z = np.load(f)
        add('kx_' + os.path.basename(f)[:-4], z['oof'], z['test'], 'kratosyan10f')
def l_hg():
    d = find_dir('single-catboost-eda-ps6e10')
    add('hg_cat', np.load(glob.glob(f'{d}/**/oof_train.npy', recursive=True)[0]),
        np.load(glob.glob(f'{d}/**/oof_test.npy', recursive=True)[0]), 'hermengardo')
def l_dp():
    d = find_dir('s6e10-numeric-keys-catboost-ctr')
    o = pd.read_csv(glob.glob(f'{d}/**/oof_predictions.csv', recursive=True)[0])
    tp = np.zeros(len(te)); n = 0
    for f in sorted(glob.glob(f'{d}/**/catboost_numeric_ctr_fold*.npz', recursive=True)):
        z = np.load(f)
        assert (z['test_ids'] == te['id'].values).all()
        tp += z['test_pred']; n += 1
    assert n == 5, n
    add('dp_catctr', by_id(o, tr['id'], 'catboost_numeric_ctr'), tp / n, 'denpugovkin')
def l_yk():
    d = find_dir('ps-s6-e10-realmlp-pytabkit')
    add('yk_realmlp', by_id(pd.read_csv(f'{d}/oof_preds.csv'), tr['id'], 'satisfaction'),
        by_id(pd.read_csv(f'{d}/submission.csv'), te['id'], 'satisfaction'), 'yekenot')


# Sadam Torres artifacts (38 members + 2 original-data teacher predictions)
def l_st():
    d = find_dir('ps-s6e10-airline-satisfaction-artifacts')
    for f in sorted(glob.glob(f'{d}/**/preds_*.npz', recursive=True)):
        z = np.load(f)
        add('st_' + os.path.basename(f)[6:-4], z['oof'], z['test'], 'sadamtorres')
    for f in ['orig_cat_pred', 'orig_model_pred']:
        z = np.load(glob.glob(f'{d}/**/{f}.npz', recursive=True)[0])
        add('st_' + f, z['train'], z['test'], 'orig teacher')
# Aman Atar original-prior stack v5 (5 base members)
def l_am():
    d = find_dir('s6e10-the-original-prior-stack-v5')
    z = np.load(glob.glob(f'{d}/**/oof_artifacts.npz', recursive=True)[0])
    for k in ['lgbm_champion', 'lgbm_te', 'lgbm_te_rows', 'xgb_gpu', 'catboost_gpu']:
        add('am_' + k, z['oof_' + k], z['test_' + k], 'amanatar')
# Ahmet Kagan Koral (5 members)
def l_kk():
    d = find_dir('s6e10-encodings-catboost-realmlp-lb-0-961')
    o, t = pd.read_csv(f'{d}/oof_members.csv'), pd.read_csv(f'{d}/test_members.csv')
    assert (o['id'].values == tr['id'].values).all() and (t['id'].values == te['id'].values).all()
    for c in o.columns[1:]:
        add('kk_' + c, o[c].values, t[c].values, 'kagankoral')
# stack-level OOFs: busyaprime's stack (LB 0.96169) and najiama blends 02/03
def l_stacklevel():
    d = find_dir('s6e10-route-or-distance-both-lb-0-96169')
    z = np.load(glob.glob(f'{d}/**/oof_stacks.npz', recursive=True)[0])
    add('bp_stack53', z['oof53'], z['test53'], 'stacklevel')
    d = find_dir('s6e10-oof')
    for k in ['02', '03']:
        add('nj_' + k, by_id(pd.read_csv(f'{d}/{k}_blend_oof.csv'), tr['id'], 'satisfaction'),
            by_id(pd.read_csv(f'{d}/{k}_submission.csv'), te['id'], 'satisfaction'), 'stacklevel')

# Chris Deotte: 9 tabular foundation models + tuned XGBoost / RealMLP (SKF5 rs42, train/test row order)
def l_cd():
    d = find_dir('s6e10-tfm-oof-and-test-predictions')
    for f in sorted(glob.glob(f'{d}/**/oof_*.npy', recursive=True)):
        m = os.path.basename(f)[4:-4]
        add('cd_' + m, np.load(f), np.load(f.replace('oof_', 'test_')), 'cdeotte tfm')

# Every source is optional: if an author re-runs or removes a file, that library is skipped (and reported) instead of
# breaking the notebook. Kernel outputs are attached at their latest version, so the member count can change over time.
SKIPPED = []
for fn in [l_gp, l_mg, l_gl, l_ar, l_sa, l_nj, l_pfn, l_mi, l_wx, l_th, l_bp, l_ab, l_kx, l_hg, l_dp, l_yk,
           l_st, l_am, l_kk, l_stacklevel, l_cd]:
    before = dict(OOF)
    try:
        fn()
    except Exception as e:
        for k in set(OOF) - set(before):   # roll back a partially loaded library
            OOF.pop(k); TEST.pop(k); SRC.pop(k)
        SKIPPED.append(fn.__name__)
        print(f'skipped {fn.__name__}: {type(e).__name__}: {e}')
names = list(OOF)
print(len(names), 'members |', pd.Series(SRC).value_counts().to_dict())
print(f'loaded in {time.time() - T0:.0f}s; skipped libraries: {SKIPPED}')

skipped l_dp: AssertionError: 0
202 members | {'goodpjw P8': 48, 'sadamtorres': 38, 'sachith7': 24, 'golem': 17, 'megayak': 11, 'cdeotte tfm': 11, 'mitudru': 7, 'kagankoral': 5, 'amanatar': 5, 'busyaprime': 5, 'arhancanli12': 4, 'megayak honest': 4, 'TabPFN outputs': 3, 'stacklevel': 3, 'thisray': 3, 'abdullah pfn': 3, 'kratosyan10f': 2, 'wangxintong111': 2, 'abdullah gbdt': 2, 'orig teacher': 2, 'najiama': 1, 'hermengardo': 1, 'yekenot': 1}
loaded in 50s; skipped libraries: ['l_dp']


In [3]:
single = pd.Series({n: roc_auc_score(y, OOF[n]) for n in names}).sort_values(ascending=False)
best = single.groupby(pd.Series(SRC)).agg(['size', 'max']).sort_values('max', ascending=False)
best.columns = ['members', 'best single OOF AUC']
best

,members,best single OOF AUC
stacklevel,3,0.961989
najiama,1,0.961744
abdullah pfn,3,0.961596
sachith7,24,0.961413
abdullah gbdt,2,0.961396
golem,17,0.961388
goodpjw P8,48,0.961345
mitudru,7,0.961307
TabPFN outputs,3,0.961268
sadamtorres,38,0.961253


## 3. Nested logistic-regression stack on logits
`LogisticRegression(C=0.003)` on clipped logits of all 199 members (with ~200 correlated inputs a much stronger L2 penalty than the default works best: C=1 0.96210, 0.01 0.96212, **0.003 0.96213**, 0.001 0.96213 on 187 members). Each row's stack prediction comes from a stacker that never saw that row
(meta folds `StratifiedKFold(5, shuffle=True, random_state=7)`); the test prediction uses one stacker fitted on all OOF rows.

In [4]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

Z = np.column_stack([logit(OOF[n]) for n in names])
ZT = np.column_stack([logit(TEST[n]) for n in names])
META = list(StratifiedKFold(5, shuffle=True, random_state=7).split(Z, y))
def report(tag, o):
    print(f'{tag:22s} nested CV {roc_auc_score(y, o):.6f}  folds ' + ' '.join(f'{roc_auc_score(y[b], o[b]):.5f}' for _, b in META), flush=True)

t = time.time()
lr_oof = np.zeros(NTR)
for a, b in META:
    lr_oof[b] = LogisticRegression(C=0.003, max_iter=3000).fit(Z[a], y[a]).decision_function(Z[b])
lr_test = LogisticRegression(C=0.003, max_iter=3000).fit(Z, y).decision_function(ZT)
report(f'LR stack ({len(names)})', lr_oof); print(f'({time.time() - t:.0f}s)')

LR stack (202)         nested CV 0.962171  folds 0.96283 0.96169 0.96188 0.96209 0.96237
(292s)


## 4. LightGBM stacker
A shallow, heavily regularised LightGBM on the same logits (here the single-model members of section 2's first two blocks, without the
extra libraries, stack-level OOFs and TFM members). Early stopping uses a 10% split of each meta-training part only, so the meta-validation fold stays untouched.

In [5]:
core = [i for i, n in enumerate(names) if SRC[n] not in {'sadamtorres', 'orig teacher', 'amanatar', 'kagankoral', 'stacklevel', 'cdeotte tfm'}]
P = dict(objective='binary', learning_rate=0.03, num_leaves=15, min_data_in_leaf=2000, feature_fraction=0.5,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, verbose=-1, num_threads=4, seed=0)
t = time.time()
A, B = Z[:, core], ZT[:, core]
gb_oof, its = np.zeros(NTR), []
for a, b in META:
    ai = np.random.default_rng(0).permutation(a); cut = int(len(ai) * 0.9)
    m = lgb.train(P, lgb.Dataset(A[ai[:cut]], y[ai[:cut]]), 3000, valid_sets=[lgb.Dataset(A[ai[cut:]], y[ai[cut:]])],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    its.append(m.best_iteration); gb_oof[b] = m.predict(A[b], raw_score=True, num_iteration=m.best_iteration)
gb_test = lgb.train(P, lgb.Dataset(A, y), int(np.mean(its) * 1.1)).predict(B, raw_score=True)
report(f'LightGBM stacker ({len(core)})', gb_oof); print(f'iterations {its} ({time.time() - t:.0f}s)')

LightGBM stacker (138) nested CV 0.962010  folds 0.96268 0.96153 0.96173 0.96196 0.96223
iterations [224, 254, 221, 275, 247] (199s)


## 5. Blend and submit
Both stackers output logits on the same scale, so they are averaged directly. The table shows every blend weight; **v3 submits W = 0 (plain LR)**.

In [6]:
for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    report(f'LR*{1 - w:.1f} + LGB*{w:.1f}', (1 - w) * lr_oof + w * gb_oof)
W = 0.0   # v3: submit the plain LR stack (see the intro); the table above shows the LightGBM blend for comparison
pred = 1 / (1 + np.exp(-((1 - W) * lr_test + W * gb_test)))
sub = pd.DataFrame({'id': te['id'], 'satisfaction': pred})
sub.to_csv('submission.csv', index=False)
print(sub.shape, sub['satisfaction'].describe().round(4).to_dict())
print(f'total {time.time() - T0:.0f}s')
sub.to_csv('submission_cv_validated.csv', index=False)   # v4 recipe, kept for comparison

LR*1.0 + LGB*0.0       nested CV 0.962171  folds 0.96283 0.96169 0.96188 0.96209 0.96237
LR*0.9 + LGB*0.1       nested CV 0.962178  folds 0.96284 0.96170 0.96188 0.96210 0.96238
LR*0.8 + LGB*0.2       nested CV 0.962180  folds 0.96285 0.96170 0.96188 0.96211 0.96239
LR*0.7 + LGB*0.3       nested CV 0.962177  folds 0.96284 0.96170 0.96188 0.96211 0.96239
LR*0.6 + LGB*0.4       nested CV 0.962168  folds 0.96284 0.96169 0.96187 0.96210 0.96238
LR*0.5 + LGB*0.5       nested CV 0.962155  folds 0.96282 0.96167 0.96186 0.96209 0.96237
(299844, 2) {'count': 299844.0, 'mean': 0.4436, 'std': 0.4333, 'min': 0.0054, '25%': 0.0352, '50%': 0.1688, '75%': 0.9466, 'max': 0.9979}
total 599s


## 6. Blending with the best public stack
The final submission rank-averages two public outputs:
[samanyu1808 · Megastack](https://www.kaggle.com/code/samanyu1808/s6e10-megastack-please-upvote) (its `submission_lr_only.csv`, public 0.96184) with weight 0.95, and
[Trust the CV](https://www.kaggle.com/code/yeashusemwal/trust-the-cv) (209 members, 0.8 × LR + 0.2 × LightGBM stacker, nested CV 0.962193) with weight 0.05.
On the public LB, adding more of our CV-chosen stacks to the Megastack output lowered the score step by step (weight 0.05: 0.96184, 0.1: 0.96183, 0.3: 0.96183, 0.5 and 0.7: 0.96182),
so the weight was set from those results. A missing source is skipped and the weights are renormalised.

In [7]:
PUBLIC = [('s6e10-megastack-please-upvote', 'submission_lr_only.csv', 0.95, 'samanyu1808 Megastack (LR only)'),
          ('trust-the-cv', 'submission.csv', 0.05, 'Trust the CV')]
parts = []
for slug, fname, w, name in PUBLIC:
    try:
        f = glob.glob(f'{find_dir(slug)}/**/{fname}', recursive=True)[0]
        p = pd.read_csv(f).set_index('id').loc[te['id'], 'satisfaction']
        parts.append((w, p.rank(pct=True).values, name))
    except Exception as e:
        print(f'skipped {name}: {type(e).__name__}: {e}')
if parts:
    wsum = sum(w for w, _, _ in parts)
    blend = sum(w * r for w, r, _ in parts) / wsum
    blend = (blend - blend.min()) / (blend.max() - blend.min())   # rescale to [0, 1]; ranks unchanged
    print('blend weights:', [(n, round(w / wsum, 3)) for w, _, n in parts])
    print('rank correlation with the plain LR stack: %.6f' % pd.Series(blend).corr(sub['satisfaction'], method='spearman'))
    pd.DataFrame({'id': te['id'], 'satisfaction': blend}).to_csv('submission.csv', index=False)
print(f'total {time.time() - T0:.0f}s')

blend weights: [('samanyu1808 Megastack (LR only)', 0.95), ('Trust the CV', 0.05)]
rank correlation with the plain LR stack: 0.999848
total 602s


## Conclusion

Note: the section 6 blend weight comes from public-LB results rather than cross-validation, so for a purely CV-chosen submission use submission_cv_validated.csv (the v4 recipe) or the [Trust the CV](https://www.kaggle.com/code/yeashusemwal/trust-the-cv) notebook.

- The public OOF libraries are saturated: 50+ extra GBDT/NN members on the same features add ≈0 nested AUC.
- What still helps is **diversity of model family**: tabular foundation models (v2: +0.00003 on top of 187 members), 15-fold TabPFN-3.5 members, and a non-linear (LightGBM) meta-learner blended with the linear stack.
- A strong L2 penalty (C=0.003) beats sklearn's default C=1 by ≈0.00003 with ~200 correlated inputs.
- A blend weight tuned on the OOF buys about 0.00001 nested CV here, below the noise; the untuned plain LR is the safer submission.
- Select by nested CV: five variants of this stack spanned only 0.00005 on the public LB, well inside its noise.

If this was useful, an upvote is appreciated, and please upvote the source notebooks and datasets, which did the real work.